# Raw AFL event data exploration

This notebook describes the schema, event ordering, team attribution and
shot annotations in the **2023 torpdata snapshot**. It provides a reproducible
inspection of the source data and identifies processing requirements for
possession-based sequence modelling. The season falls within the proposed
training period.

Data source: [torpdata chains release](https://github.com/peteowen1/torpdata/releases/tag/chains-data).
Installation and launch instructions are in the repository README. Run cells
in order with **Shift+Enter** using the project kernel. The sections document
the inspection methods, results and implications for preprocessing.

`events` contains the loaded source table. Filters, summaries and display
columns are created separately. The notebook leaves the Parquet file
unchanged and performs no model fitting or training-example construction.

## 1. Environment and project paths

The setup imports pandas, the Jupyter display function and the package's
`download_season` function. It locates the project root by searching the
kernel's working directory and its parents for `pyproject.toml` and the
source manifest. This supports execution from the repository root or
`notebooks/` without hard-coded local paths or changes to `sys.path`.

The environment check reports Python and pandas versions and verifies that
the kernel uses the repository's `.venv`. If the check is false, select
**Python (afl-opposition-analysis)**. Kernel registration instructions are
included in the README.

In [5]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

from afl_opposition.data import download_season

working_directory = Path.cwd().resolve()
candidates = [working_directory, *working_directory.parents]
project_root = next(
    (path for path in candidates
     if (path / "pyproject.toml").is_file()
     and (path / "data" / "sources.json").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Launch this notebook from the project root or notebooks folder.")

print("Python:", sys.version.split()[0])
print("Using project .venv:", Path(sys.prefix).resolve() == project_root / ".venv")
print("pandas:", pd.__version__)

Python: 3.14.7
Using project .venv: True
pandas: 3.0.6


## 2. Snapshot verification and loading

`download_season` validates the file size and SHA-256 checksum against
`data/sources.json`. A matching local file is reused; an absent file is
downloaded to `data/raw/`. This inspection uses the manifest entry for 2023.

`pd.read_parquet` loads the file into a DataFrame using PyArrow.
`dtype_backend="numpy_nullable"` preserves nullable integers and booleans,
including missing shot flags. The printed shape gives `(rows, columns)`.
All columns are loaded for inspection; feature selection is a separate
modelling step.

`redirect_stdout` captures the downloader's status messages so the
display contains only the snapshot filename and table shape. Local
directory paths are not included in the displayed result.


In [6]:
from contextlib import redirect_stdout
from io import StringIO

SEASON = 2023
with redirect_stdout(StringIO()):
    data_path = download_season(
        SEASON,
        manifest_path=project_root / "data" / "sources.json",
        output_dir=project_root / "data" / "raw",
    )
events = pd.read_parquet(
    data_path,
    engine="pyarrow",
    dtype_backend="numpy_nullable",
)

print("Snapshot:", data_path.name)
print("DataFrame shape:", events.shape)

Snapshot: chains_data_2023_all.parquet
DataFrame shape: (442482, 63)


## 3. Dataset size and identifiers

A row represents an event annotation. Matches contain multiple rows, and
each raw chain is identified by the combination of match, quarter and
chain number. Chain numbers alone are not globally unique.

`nunique()` counts distinct values, while `drop_duplicates()` counts
distinct key combinations. The raw event-key check also includes
`display_order`.

This snapshot contains 442,482 rows, 216 matches and 54,418 raw chains.
Unique event keys do not establish that every row is a separate physical
action. Rows are also correlated within matches and should not be treated
as independent training examples.

In [7]:
chain_key = ["match_id", "period", "chain_number"]
event_key = [*chain_key, "display_order"]

overview = pd.Series({
    "Rows": len(events),
    "Columns": len(events.columns),
    "Matches": events["match_id"].nunique(),
    "Raw chains": len(events[chain_key].drop_duplicates()),
    "Action descriptions": events["description"].nunique(),
    "Extra rows sharing an event key": int(events.duplicated(event_key).sum()),
    "Rows with missing key fields": int(events[event_key].isna().any(axis=1).sum()),
}, name="count")
display(overview.to_frame())
print("Date range:", events["date"].min(), "to", events["date"].max())

,count
Rows,442482
Columns,63
Matches,216
Raw chains,54418
Action descriptions,77
Extra rows sharing an event key,0
Rows with missing key fields,0


Date range: 2023-03-16 to 2023-09-30


## 4. Schema and missing values

`dtypes` reports the loaded representation of each column.
`isna().sum()` counts missing entries; dividing by the row count gives
missingness as a percentage. The temporary display setting shows all
63 columns without modifying the table.

| Fields | Meaning |
| --- | --- |
| `match_id`, `period`, `display_order` | Match, quarter and annotation order |
| `period_seconds` | Time within the quarter |
| `description` | Categorical action annotation |
| `team_id`, `player_id` | Acting team and player where supplied |
| `chain_number`, `chain_team_id` | Upstream chain and its assigned team |
| `x`, `y` | Numeric event coordinates, recorded as integers |
| `shot_at_goal` | Explicit shot marker on the current row |
| `initial_state`, `final_state` | Upstream chain start and end categories |

Missing team and player IDs occur in annotations that may identify useful
stoppages or boundaries; these rows require interpretation before removal.
Entirely missing columns may have misleading inferred types. For example,
the boolean dtype of empty `venue_local_start_time` does not establish its
intended type. Chain outcomes and final match scores contain future
information and must be excluded from prefix features.

In [8]:
missing_counts = events.isna().sum()
schema = pd.DataFrame({
    "dtype": events.dtypes.astype(str),
    "missing_rows": missing_counts,
    "missing_percent": (100 * missing_counts / len(events)).round(2),
}).rename_axis("column")

with pd.option_context("display.max_rows", 100):
    display(schema)

,dtype,missing_rows,missing_percent
column,,,
display_order,Int32,0,0.00
description,string,0,0.00
period_seconds,Int32,0,0.00
player_id,string,27844,6.29
team_id,string,27844,6.29
disposal,string,285921,64.62
shot_at_goal,boolean,431594,97.54
behind_info,string,438824,99.17
x,Int32,0,0.00


## 5. Action vocabulary

`value_counts(dropna=False)` counts action descriptions and includes
missing values if present. The output shows the 20 most frequent
annotations and their share of all rows. These are annotation frequencies,
not possession frequencies or scoring probabilities.

A *handball* passes the ball by punching it from the other hand. A *mark*
records a catch of a qualifying kick. A *spoil* disrupts an attempted
catch. *F50* refers to the attacking area within 50 metres of goal.
`Kick Into F50` can accompany a `Kick` annotation and must not automatically
be counted as another kick. Further terminology is covered in the
[AFL statistics glossary](https://www.afl.com.au/news/144837/stats-glossary-every-stat-explained).

Action categories are candidates for a future embedding vocabulary after
overlapping annotations have been reconciled.

In [9]:
action_counts = events["description"].value_counts(dropna=False).rename("rows").to_frame()
action_counts["percent_of_rows"] = (100 * action_counts["rows"] / len(events)).round(2)

display(action_counts.head(20))
print("Distinct non-missing descriptions:", events["description"].nunique())
# Use display(action_counts) to inspect the complete vocabulary.

,rows,percent_of_rows
description,,
Kick,89487,20.22
Handball,64045,14.47
Handball Received,51143,11.56
Uncontested Mark,36863,8.33
Loose Ball Get,27015,6.11
Kick Into F50,19780,4.47
Kick Inside 50 Result,19771,4.47
Spoil,14306,3.23
Loose Ball Get Crumb,13417,3.03


Distinct non-missing descriptions: 77


## 6. Match and quarter selection

Fixture metadata is repeated across event rows. Selecting the fixture
columns and removing duplicate combinations produces a match catalogue.
The first ten fixtures are displayed with their IDs.

`MATCH_ID` and `QUARTER` control the subsequent passage inspection. The
default is Richmond versus Carlton on 16 March 2023, quarter one. After
changing either parameter, rerun this cell and subsequent cells. Quarter
numbers and quarter time are local to each match.

In [10]:
match_columns = ["match_id", "date", "home_team_team_name", "away_team_team_name"]
matches = (
    events[match_columns]
    .drop_duplicates()
    .sort_values(["date", "match_id"])
    .reset_index(drop=True)
)
display(matches.head(10))

MATCH_ID = "CD_M20230140101"
QUARTER = 1
selected_match = matches.loc[matches["match_id"].eq(MATCH_ID)]
if selected_match.empty:
    raise ValueError("Choose a MATCH_ID from the matches table.")
display(selected_match)

,match_id,date,home_team_team_name,away_team_team_name
0,CD_M20230140101,2023-03-16,Richmond,Carlton
1,CD_M20230140102,2023-03-17,Geelong Cats,Collingwood
2,CD_M20230140103,2023-03-18,North Melbourne,West Coast Eagles
3,CD_M20230140104,2023-03-18,Port Adelaide,Brisbane Lions
4,CD_M20230140105,2023-03-18,Melbourne,Western Bulldogs
5,CD_M20230140106,2023-03-18,Gold Coast SUNS,Sydney Swans
6,CD_M20230140107,2023-03-19,GWS GIANTS,Adelaide Crows
7,CD_M20230140108,2023-03-19,Hawthorn,Essendon
8,CD_M20230140109,2023-03-19,St Kilda,Fremantle
9,CD_M20230140201,2023-03-23,Carlton,Geelong Cats


,match_id,date,home_team_team_name,away_team_team_name
0,CD_M20230140101,2023-03-16,Richmond,Carlton


## 7. Ordered event table

The passage table contains a copy of the selected match-quarter, sorted by
`display_order`. A lookup maps acting and chain team IDs to readable names. These
display columns are added to the copy, leaving `events` unchanged. Missing
acting teams remain missing rather than inheriting the assigned chain team.

The table shows annotation order, raw chain, quarter time, action, acting
team, chain team, coordinates and the current-row shot flag. `ROWS_TO_SHOW`
limits the output to 25 rows by default. Event annotations describe
individual observations, not video frames or the positions of every player.

In [11]:
team_names = (
    events[["team_id", "team_team_name"]]
    .dropna()
    .drop_duplicates("team_id")
    .set_index("team_id")["team_team_name"]
)
quarter = events.loc[
    events["match_id"].eq(MATCH_ID) & events["period"].eq(QUARTER)
].sort_values("display_order").copy()
if quarter.empty:
    raise ValueError("No events found for this match and quarter.")

quarter["acting_team"] = quarter["team_id"].map(team_names)
quarter["chain_team"] = quarter["chain_team_id"].map(team_names)
event_columns = [
    "display_order", "chain_number", "period_seconds", "description",
    "acting_team", "chain_team", "x", "y", "shot_at_goal",
]
ROWS_TO_SHOW = 25
print("Rows in selected quarter:", len(quarter))
display(quarter[event_columns].head(ROWS_TO_SHOW))

Rows in selected quarter: 587


,display_order,chain_number,period_seconds,description,acting_team,chain_team,x,y,shot_at_goal
0,1,1,0,Centre Bounce,<NA>,Richmond Tigers,0,0,<NA>
1,2,1,11,Hard Ball Get,Richmond Tigers,Richmond Tigers,-2,1,<NA>
2,3,1,11,Handball,Richmond Tigers,Richmond Tigers,-2,0,<NA>
3,4,1,15,Ball Up Call,<NA>,Richmond Tigers,-3,0,<NA>
4,5,2,24,Free For,Richmond Tigers,Richmond Tigers,-3,-2,<NA>
5,6,2,26,Kick,Richmond Tigers,Richmond Tigers,-8,-7,<NA>
6,7,2,26,Kick Into F50,Richmond Tigers,Richmond Tigers,-8,-7,<NA>
7,8,2,27,Kick Inside 50 Result,<NA>,Richmond Tigers,42,-4,<NA>
8,9,2,29,Mark Fumbled,Carlton Blues,Richmond Tigers,42,-4,<NA>
9,10,2,29,Mark Dropped,Carlton Blues,Richmond Tigers,43,-4,<NA>


## 8. Event timing

`diff()` subtracts the preceding row's quarter time within the selected
match-quarter. The first gap is missing because no predecessor exists.
Time differences must not be calculated across quarter boundaries.

Irregular gaps and tied timestamps are expected in event data. A zero gap
can indicate overlapping annotations or distinct events recorded at the
same second. `display_order` preserves the recorded ordering when times
tie; timestamps alone are insufficient for deduplication. A sequence model
can represent elapsed time explicitly rather than assuming a fixed
interval between rows.

In [12]:
quarter["gap_seconds"] = quarter["period_seconds"].diff()
time_checks = pd.Series({
    "Pairs with a preceding timestamp": int(quarter["gap_seconds"].notna().sum()),
    "Pairs sharing a timestamp": int(quarter["gap_seconds"].eq(0).sum()),
    "Time decreases": int(quarter["gap_seconds"].lt(0).sum()),
}, name="count")
display(time_checks.to_frame())
display(quarter[["display_order", "period_seconds", "gap_seconds", "description"]].head(20))

,count
Pairs with a preceding timestamp,586
Pairs sharing a timestamp,158
Time decreases,0


,display_order,period_seconds,gap_seconds,description
0,1,0,<NA>,Centre Bounce
1,2,11,11,Hard Ball Get
2,3,11,0,Handball
3,4,15,4,Ball Up Call
4,5,24,9,Free For
5,6,26,2,Kick
6,7,26,0,Kick Into F50
7,8,27,1,Kick Inside 50 Result
8,9,29,2,Mark Fumbled
9,10,29,0,Mark Dropped


## 9. Raw chain inspection

Grouping the selected quarter by chain number gives row counts, time spans
and the number of distinct acting teams. Missing team IDs are excluded from
the distinct-team count. Set `CHAIN_NUMBER` to an entry in this table to
display its annotations. `CHAIN_ROWS_TO_SHOW` caps the output at 40 rows.

In the default fixture, chain 2 contains `Kick` and `Kick Into F50` at
26 seconds, followed by `Kick Inside 50 Result`. The kick annotations
share a time and location, illustrating the need to reconcile annotations
before counting physical actions. Opposing-team fumbles also occur within
the same raw chain. This example illustrates the data structure; it does
not establish a recurring team strategy.

In [13]:
chain_overview = quarter.groupby("chain_number", sort=True).agg(
    rows=("description", "size"),
    start_seconds=("period_seconds", "min"),
    end_seconds=("period_seconds", "max"),
    distinct_acting_teams=("team_id", "nunique"),
)
display(chain_overview.head(15))

CHAIN_NUMBER = 2
CHAIN_ROWS_TO_SHOW = 40
selected_chain = quarter.loc[quarter["chain_number"].eq(CHAIN_NUMBER)]
if selected_chain.empty:
    raise ValueError("Choose a CHAIN_NUMBER from chain_overview for this quarter.")
print("Rows in selected chain:", len(selected_chain))
display(selected_chain[event_columns].head(CHAIN_ROWS_TO_SHOW))

,rows,start_seconds,end_seconds,distinct_acting_teams
chain_number,,,,
1,4,0,15,1
2,9,24,35,2
3,13,84,108,1
4,8,155,169,2
5,3,169,175,2
6,7,175,187,2
7,7,187,216,2
8,9,216,228,2
9,5,228,241,2


Rows in selected chain: 9


,display_order,chain_number,period_seconds,description,acting_team,chain_team,x,y,shot_at_goal
4,5,2,24,Free For,Richmond Tigers,Richmond Tigers,-3,-2,<NA>
5,6,2,26,Kick,Richmond Tigers,Richmond Tigers,-8,-7,<NA>
6,7,2,26,Kick Into F50,Richmond Tigers,Richmond Tigers,-8,-7,<NA>
7,8,2,27,Kick Inside 50 Result,<NA>,Richmond Tigers,42,-4,<NA>
8,9,2,29,Mark Fumbled,Carlton Blues,Richmond Tigers,42,-4,<NA>
9,10,2,29,Mark Dropped,Carlton Blues,Richmond Tigers,43,-4,<NA>
10,11,2,32,Loose Ball Get Crumb,Richmond Tigers,Richmond Tigers,46,-5,<NA>
11,12,2,32,Kick,Richmond Tigers,Richmond Tigers,52,-5,True
12,13,2,35,Goal,Richmond Tigers,Richmond Tigers,51,-5,<NA>


## 10. Acting team and chain attribution

The attribution check compares `team_id` and `chain_team_id` where both are
known. A boolean mask identifies differing-team rows; grouping their keys
counts the affected raw chains. The second table summarises their most common action types.

The snapshot contains 48,231 differing-team rows across 33,329 raw chains.
Defensive actions such as spoils can produce mismatches without a change
of possession. Nevertheless, assigning every action to `chain_team_id`
would misattribute these rows. Possession construction requires rules that
distinguish interference from an opponent gaining control.

In [14]:
known_teams = events["team_id"].notna() & events["chain_team_id"].notna()
different_team = (
    known_teams & events["team_id"].ne(events["chain_team_id"])
).fillna(False)

ownership_checks = pd.Series({
    "Rows with both team IDs known": int(known_teams.sum()),
    "Rows with different acting and chain teams": int(different_team.sum()),
    "Raw chains containing those rows": len(events.loc[different_team, chain_key].drop_duplicates()),
}, name="count")
display(ownership_checks.to_frame())
display(
    events.loc[different_team, "description"]
    .value_counts()
    .head(12)
    .rename("differing_team_rows")
    .to_frame()
)

,count
Rows with both team IDs known,414638
Rows with different acting and chain teams,48231
Raw chains containing those rows,33329


,differing_team_rows
description,
Spoil,12030
Loose Ball Get,8774
Uncontested Mark,5035
Loose Ball Get Crumb,4654
Hard Ball Get,4120
Handball,3133
Contested Mark,2004
Gather from Opposition,1198
Handball Received,1007


## 11. Shot annotations and target definition

`value_counts(dropna=False)` includes missing shot flags. The mask
`eq(True).fillna(False)` selects explicitly flagged shots for inspection;
it neither overwrites the source flags nor creates negative training labels.

The snapshot has 10,888 True flags and 431,594 missing values, with no
explicit False values. Flagged annotations are `Kick` or `Ground Kick`.
Fifteen have an acting team different from the chain team. The final table
displays up to 20 flagged shots in the selected match-quarter.

The proposed prediction target is whether the same team takes a subsequent
shot before its possession ends, given an observed pre-shot prefix. This
differs from classifying the current row as a shot. Validated possession
boundaries are required to assign labels. Full-chain outcomes and final
scores must not enter the model's input prefix.

In [15]:
display(events["shot_at_goal"].value_counts(dropna=False).rename("rows").to_frame())
explicit_shots = events["shot_at_goal"].eq(True).fillna(False)
display(
    events.loc[explicit_shots, "description"]
    .value_counts()
    .rename("flagged_shot_rows")
    .to_frame()
)
print("Flagged shots with different acting and chain teams:", int((explicit_shots & different_team).sum()))

quarter_shots = quarter.loc[quarter["shot_at_goal"].eq(True).fillna(False)]
display(quarter_shots[event_columns].head(20))

,rows
shot_at_goal,
<NA>,431594
True,10888


,flagged_shot_rows
description,
Kick,10663
Ground Kick,225


Flagged shots with different acting and chain teams: 15


,display_order,chain_number,period_seconds,description,acting_team,chain_team,x,y,shot_at_goal
11,12,2,32,Kick,Richmond Tigers,Richmond Tigers,52,-5,True
24,25,3,107,Kick,Carlton Blues,Carlton Blues,60,-18,True
148,149,18,455,Kick,Carlton Blues,Carlton Blues,64,-24,True
193,194,22,613,Kick,Richmond Tigers,Richmond Tigers,66,16,True
217,218,26,669,Kick,Richmond Tigers,Richmond Tigers,69,21,True
250,251,32,747,Kick,Richmond Tigers,Richmond Tigers,46,10,True
321,322,41,921,Kick,Carlton Blues,Carlton Blues,51,6,True
352,353,44,1005,Kick,Richmond Tigers,Richmond Tigers,63,-24,True
447,448,56,1244,Kick,Carlton Blues,Carlton Blues,52,0,True
472,473,60,1378,Kick,Richmond Tigers,Richmond Tigers,30,-27,True


## 12. Coordinates and spatial interpretation

The spatial summary reports coordinate ranges, means and standard deviations,
alongside distinct venue dimensions. Although `x` and `y` are
stored as integers, they represent spatial quantities. A sequence model
can combine numeric spatial inputs with categorical action embeddings.

Raw x spans -85 to 86 and y spans -71 to 71 in this file. These ranges do
not establish attack direction. Spatial features require an audit of the
coordinate frame and its relationship to the acting team, assigned chain
team and quarter. Event locations do not provide continuous trajectories,
off-ball positions or guaranteed kick reception endpoints.

In [16]:
coordinate_summary = events[["x", "y"]].agg(["min", "max", "mean", "std"]).T.round(2)
display(coordinate_summary)
venue_dimensions = (
    events[["venue_name", "venue_length", "venue_width"]]
    .drop_duplicates()
    .sort_values("venue_name")
)
display(venue_dimensions.head(10))

,min,max,mean,std
x,-85.0,86.0,2.61,40.09
y,-71.0,71.0,0.27,32.80


,venue_name,venue_length,venue_width
80237,Adelaide Hills,152,111
5963,Adelaide Oval,167,123
96753,Corroboree Group Oval Manuka,162,138
12286,ENGIE Stadium,164,128
98828,GMHBA Stadium,170,115
20690,Gabba,156,138
0,MCG,160,141
170694,Mars Stadium,160,129
3945,Marvel Stadium,160,129
154193,Ninja Stadium,160,124


## Interpretation and modelling requirements

The dataset combines discrete action annotations, numeric coordinates,
irregular event times and repeated fixture metadata. Its raw rows and
chain IDs require processing before they can represent possession sequences.

That processing requires rules for overlapping annotations, control
changes, stoppages and coordinate orientation, supported by manual audits
of varied passages. Missing shot flags alone cannot define negative
possession labels. Prefix features must contain only information available
at the prediction cutoff.

A complete run from a restarted kernel reproduces the inspection from the
recorded snapshot. Downloaded data remains in the Git-ignored `data/raw/`
directory. Saved outputs provide a reference for the recorded snapshot.